# Normalize Bulgarian text without losing punctuation

This uses the production dual normalizer. It writes word-only text for alignment and punctuation-preserving text for training; both views are guaranteed to contain exactly the same words.

In [ ]:
import re, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'testSplit' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from bulgarian_normalization import normalize_for_mfa, normalize_with_punctuation

SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?…])\s+')

def normalise_to_sentences(raw_text):
    pairs = []
    for line in raw_text.splitlines():
        for chunk in SENTENCE_SPLIT_RE.split(line.strip()):
            words = normalize_for_mfa(chunk)
            punctuated = normalize_with_punctuation(chunk)
            if words:
                assert normalize_for_mfa(punctuated) == words
                pairs.append((words, punctuated))
    return pairs

In [ ]:
sample = '«Шибил». Здравей, свят! Имам 3 ябълки и 5 круши?'
for words, punctuated in normalise_to_sentences(sample):
    print('MFA:', words)
    print('TTS:', punctuated)

## Normalize a source file

The alignment file is suitable for aeneas/MFA. The prosody file is the only version that should be copied into training metadata.

In [ ]:
INPUT = Path('testText1.txt')
ALIGNMENT_OUTPUT = Path('testText1.alignment.txt')
PROSODY_OUTPUT = Path('testText1.prosody.txt')

pairs = normalise_to_sentences(INPUT.read_text(encoding='utf-8'))
ALIGNMENT_OUTPUT.write_text('\n'.join(x[0] for x in pairs) + '\n', encoding='utf-8')
PROSODY_OUTPUT.write_text('\n'.join(x[1] for x in pairs) + '\n', encoding='utf-8')
print(f'{len(pairs)} sentences -> {ALIGNMENT_OUTPUT}, {PROSODY_OUTPUT}')